# Quickstart Tutorial: Knowledge Graph-Informed Microbiome Foundation Model

This tutorial demonstrates how to:
1. Load the 1,041-token genus vocabulary and 256-dimensional R-GCN embeddings.
2. Instantiate the `MicrobiomeTransformer` architecture.
3. Extract frozen representations from example cohorts.
4. Evaluate zero-shot cross-cohort transfer with linear probing.

In [ ]:
import sys
from pathlib import Path
import torch
import pandas as pd

# Ensure package is in python path
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from microbiome_kg.transformer.config import ModelConfig
from microbiome_kg.transformer.model import MicrobiomeTransformer
from microbiome_kg.data.dataset import load_vocab, TaskDataset, collate_task_batch
from torch.utils.data import DataLoader

In [ ]:
# Load canonical vocabulary and bridge
vocab_file = str(REPO_ROOT / "data" / "vocab" / "genus_vocab.txt")
bridge_file = str(REPO_ROOT / "data" / "vocab" / "bridge.json")
rgcn_emb_file = str(REPO_ROOT / "data" / "kg" / "entity_embeddings.npy")

tok2id, id2tok = load_vocab(vocab_file)
print(f"Vocabulary size: {len(id2tok)} tokens (Special: 4, Genera: {len(id2tok)-4})")

In [ ]:
# Instantiate foundation model with Knowledge Graph initialization
cfg = ModelConfig(vocab_size=1041, d_model=256, n_heads=8, n_layers=6)
model = MicrobiomeTransformer(
    cfg,
    rgcn_emb_file=rgcn_emb_file,
    bridge_file=bridge_file,
    graph_init=True
)
print(f"Initialized MicrobiomeTransformer with {sum(p.numel() for p in model.parameters()):,} parameters.")

In [ ]:
# Load example cohort and extract representations
sample_a = str(REPO_ROOT / "data" / "examples" / "sample_cohort_a.tsv")
df_a = pd.read_csv(sample_a, sep="\t")
dataset = TaskDataset(df_a, vocab_file=vocab_file)
loader = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=collate_task_batch)

batch = next(iter(loader))
model.eval()
with torch.no_grad():
    cls_repr, seq_repr = model(batch["genus_ids"], batch["bin_ids"])

print(f"Extracted CLS representation shape: {cls_repr.shape}")
print(f"Extracted Sequence representation shape: {seq_repr.shape}")